In [3]:
import cProfile
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)


from src.poisson_hypergraph import GH
from src.NMI_func import NMI
import xgi
import numpy as np
import networkx as nx

# function that generates an intstance of the class GH with the parameters stored in true_theta and for timesteps times resulting in a graph with timesteps+1 edges
# the graph starts with 2 nodes but will have more as novel nodes are added
def generate_graph(true_theta, timesteps):
    true_p, true_q, gamma_nu, gamma_nr, gamma_eu, gamma_er = true_theta
    H = xgi.Hypergraph([[0, 1]])
    H.set_node_attributes({0 : 0, 1 : 1}, name = "label")
    g = GH(H, [0, 1], true_p, true_q)
    g.add_hyperedge(timesteps, gamma_nu, gamma_nr, gamma_eu, gamma_er)
    return g

def generate_graph_26_starting_nodes(true_theta, timesteps):
    true_p, true_q, gamma_nu, gamma_nr, gamma_eu, gamma_er = true_theta
    H = xgi.Hypergraph([[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25]])
    H.set_node_attributes({0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:1,14:1,15:1,16:1,17:1,18:1,19:1,20:1,21:1,22:1,23:1,24:1,25:1}, name="label")
    g=GH(H, [0,1], true_p, true_q)
    g.add_hyperedge(timesteps, gamma_nu, gamma_nr, gamma_eu, gamma_er)

    return g

true_p = .9
true_q = .1
# gamma_nu is poisson weight for the distribution for number of like labeled novel nodes of choosen u novel nodes added
# gamma_nr is poisson weight for the distribution for number of opposite labeled nodes of choosen u novel nodes added
# gamma_eu is poisson weight for the distribution for number of like labeled nodes of choosen u external nodes added
# gamma_er is poisson weight for the distribution for number of opposite labeled nodes of choosen u external nodes added
gamma_nu, gamma_nr, gamma_eu, gamma_er = .01, .01, 1, 0.25

# should be in this order
true_theta = [true_p, true_q, gamma_nu, gamma_nr, gamma_eu, gamma_er]
timesteps = 10

g = generate_graph_26_starting_nodes(true_theta, timesteps)

# input is an instance of the class GH defined in poisson_hypergraph.py... output a tuple of lists of total likelihood and nmi indexed by timestep
def greedy_community_detection_algo(g, generate_likelihoods):
    total_likelihoods = []
    nmis = []

    null_labels = np.random.choice([0,1], size=len(g.get_labels()))
    true_labels = g.get_labels()

    greedy_steps = 1000
    step_num = 0
    while step_num < greedy_steps:
        delta_E = 0
        e_index = np.random.choice(range(1, len(g.get_edges())))
        
        v_index = np.random.choice(list(g.get_edges()[e_index]))

        canidate_f = []
        for f_index in range(e_index):
            if len(g.get_edges()[f_index].intersection(g.get_edges()[e_index])) != 0:
                canidate_f.append(f_index)


        for f_index in canidate_f:
            delta_E += g.greedy_expectation_step_given_f(v_index, f_index, e_index, true_theta, null_labels) / len(canidate_f)
        
        if (delta_E > 0):
            null_labels[v_index] = 1 - null_labels[v_index]
            # print("swapped label of " + str(v_index) + " from edge " + str(e_index))

        step_num+=1

        # if (step_num % 25 == 0):
        if generate_likelihoods:
            total_likelihoods.append(g.total_log_likelihood(true_theta, null_labels))
            nmis.append(NMI(g.get_labels(), null_labels, g))
        
    # print("greedy labels likelihood: " + str(g.expected_log_likelihood_total(true_theta, null_labels)))
    # print("true labels likelihood: " + str(g.expected_log_likelihood_total(true_theta,true_labels)))
    # print(null_labels)
    # print(true_labels)
    if generate_likelihoods:
        return total_likelihoods, nmis
    else:
        return null_labels

def greedy_community_detection_algo_with_posterior_prob(g, generate_likelihoods):
    null_labels = np.random.choice([0,1], size=len(g.get_labels()))
    true_labels = g.get_labels()

    greedy_steps = 2000
    step_num = 0

    # TODO remove, for testing
    total_likelihoods = []
    nmis = []
    while step_num < greedy_steps:
        delta_E = 0
        e_index = np.random.choice(range(0, len(g.get_edges())))
        v_index = np.random.choice(list(g.get_edges()[e_index]))

        new_labels = null_labels.copy()
        new_labels[v_index] = 1 - new_labels[v_index]

        f_probs = g.f_prob_array_given_e(e_index, true_theta, new_labels)

        canidate_f = []
        for f_index in range(e_index):
            if len(g.get_edges()[f_index].intersection(g.get_edges()[e_index])) != 0:
                canidate_f.append(f_index)

        for f_index in canidate_f:
            delta_E += g.greedy_expectation_step_given_f(v_index, f_index, e_index, true_theta, null_labels) * f_probs[f_index]


        
        if (delta_E > 0):
            null_labels[v_index] = 1 - null_labels[v_index]
        
        # if (step_num % 25 == 0):
        if generate_likelihoods:
            total_likelihoods.append(g.total_log_likelihood(true_theta, null_labels))
            nmis.append(NMI(g.get_labels(), null_labels, g))

            
        step_num+=1

    if generate_likelihoods:
        return total_likelihoods, nmis
    else:
        return null_labels

def greedy_community_detection_algo_3_label(g, generate_likelihoods):
    NUM_ES = 3

    null_labels = np.random.choice([0,1], size=len(g.get_labels()))
    true_labels = g.get_labels()

    greedy_steps = 2000
    step_num = 0

    # TODO remove, for testing
    total_likelihoods = []
    nmis = []
    while step_num < greedy_steps:
        delta_E = 0
        
        v_index = np.random.choice(list(range(len(g.get_labels()))))

        canidate_e = []
        for e_index in range(1, len(g.get_edges())):
            if v_index in g.get_edges()[e_index]:
                canidate_e.append(e_index)

        e_indexes = np.random.choice(canidate_e, min(NUM_ES, len(canidate_e)))

        new_labels = null_labels.copy()
        new_labels[v_index] = 1 - new_labels[v_index]

        for e_index in e_indexes:
            f_probs = g.f_prob_array_given_e(e_index, true_theta, new_labels)

            canidate_f = []
            for f_index in range(e_index):
                if len(g.get_edges()[f_index].intersection(g.get_edges()[e_index])) != 0:
                    canidate_f.append(f_index)

            for f_index in canidate_f:
                delta_E += g.greedy_expectation_step_given_f(v_index, f_index, e_index, true_theta, null_labels) * f_probs[f_index]


        if (delta_E > 0):
            null_labels[v_index] = 1 - null_labels[v_index]
        
        # if (step_num % 25 == 0):
        if generate_likelihoods:
            total_likelihoods.append(g.total_log_likelihood(true_theta, null_labels))
            nmis.append(NMI(g.get_labels(), null_labels, g))

            
        step_num+=1

    if generate_likelihoods:
        return total_likelihoods, nmis
    else:
        return null_labels

In [4]:
def test():
    for _ in range(100):
        g = generate_graph(true_theta, timesteps)
        # False specifies whether to generate true log likelihood every step for graphing

        # both lines below call the different versions of the algo... 
        # 3_label has been the best in testing but takes longer



        greedy_community_detection_algo_with_posterior_prob(g, False)
        # greedy_community_detection_algo_3_label(g, False)


cProfile.run('test()', sort='ncalls')

         41010168 function calls (41010135 primitive calls) in 32.463 seconds

   Ordered by: call count

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
  7775329    1.395    0.000    1.395    0.000 {built-in method builtins.sum}
  6779016    1.191    0.000    1.191    0.000 {built-in method math.log}
6466431/6466398    0.552    0.000    0.552    0.000 {built-in method builtins.len}
  4701966    0.535    0.000    0.535    0.000 {built-in method builtins.isinstance}
  4519344    1.452    0.000    1.963    0.000 poisson_hypergraph.py:938(gammaln_fast)
  1714480    0.327    0.000    0.327    0.000 {method 'intersection' of 'set' objects}
  1453486    0.168    0.000    0.168    0.000 poisson_hypergraph.py:28(get_edges)
   554942    0.064    0.000    0.064    0.000 {method 'append' of 'list' objects}
   274131    6.789    0.000    9.404    0.000 poisson_hypergraph.py:387(log_likelihood_given_u_f)
   274131    0.069    0.000    0.069    0.000 poisson_hypergraph.py:44

KeyboardInterrupt: 

In [ ]:
print(g.f_prob_array_given_e_array(3, true_theta, g.get_labels()))

print(g.get_edges()[3])
print(g.get_edges()[0])
print(g.get_edges()[1])
print(g.get_edges()[2])

[1.75734183e-282 1.00000000e+000 2.65013245e-015]
{0, 1, 2, 3, 4, 5, 7, 8, 9, 11, 12, 19}
{0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25}
{0, 1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15}
{0, 2, 4, 6, 7, 8, 9, 10, 11, 12, 13, 15, 20}


In [ ]:
k = np.random.choice([201, 202], 100)
k2 = np.random.choice([199, 198], 100)

method1 = lambda k: g.gammaln_fast(k)
method2 = lambda k: g.gammaln_fast(k2)

%timeit method1(k)
%timeit method2(k)

2.64 µs ± 5.63 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
2.64 µs ± 9.45 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [ ]:

k = 0

g = generate_graph_26_starting_nodes(true_theta, 1000)

method1 = lambda k: g.f_prob_array_given_e(500, true_theta, g.get_labels())
method2 = lambda k: g.f_prob_array_given_e_array(500, true_theta, g.get_labels())

def test2():
    for _ in range(100000):
        g.f_prob_array_given_e_array(8, true_theta, g.get_labels())

%timeit method1(k)
%timeit method2(k)

cProfile.run('test2()', sort='ncalls')

9.91 ms ± 92.3 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
5.12 ms ± 11.4 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
         36200004 function calls in 28.743 seconds

   Ordered by: call count

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
  8600000    0.664    0.000    0.664    0.000 {method 'append' of 'list' objects}
  7200000    1.389    0.000    1.389    0.000 {built-in method builtins.sum}
  4500000    0.361    0.000    0.361    0.000 {built-in method builtins.len}
  2200000    0.681    0.000    0.681    0.000 {method 'intersection' of 'set' objects}
  1800000    0.366    0.000    0.366    0.000 {built-in method math.log}
  1700000    0.163    0.000    0.163    0.000 poisson_hypergraph.py:28(get_edges)
  1500000    0.153    0.000    0.153    0.000 {built-in method builtins.isinstance}
  1200000    1.068    0.000    1.068    0.000 {built-in method numpy.array}
  1200000    1.965    0.000    2.073    0.000 poisson_hypergraph.py

In [ ]:
true_p = .9
true_q = .1

gamma_nu, gamma_nr, gamma_eu, gamma_er = .01, .01, 1, .25

# should be in this order
true_theta = [true_p, true_q, gamma_nu, gamma_nr, gamma_eu, gamma_er]

g = generate_graph_26_starting_nodes(true_theta, 20)

print("new" + str(g.f_prob_array_given_e_array(19, true_theta, g.get_labels())))
print(g.f_prob_array_given_e(19, true_theta, g.get_labels()))

(19,)
(19,)
(19,)
(19,)
[-10.49568662  -9.33672095  -9.60842607 -15.84697929  -9.12599992
 -10.60167784  -7.41120149  -9.71378658  -9.12599992  -7.51656201
 -10.70703836  -9.71378658  -9.60842607  -4.52082973 -10.70703836
  -5.51408151  -9.68538711 -20.83514173  -4.52082973]
new[9.99886732e-04 3.18627460e-03 2.42819281e-03 4.74148006e-06
 3.93367235e-03 8.99330669e-04 2.18537353e-02 2.18537353e-03
 3.93367235e-03 1.96683617e-02 8.09397602e-04 2.18537353e-03
 2.42819281e-03 3.93367235e-01 8.09397602e-04 1.45691568e-01
 2.24832667e-03 3.23282731e-08 3.93367235e-01]
[-10.49568662  -9.33672095  -9.60842607 -15.84697929  -9.12599992
 -10.60167784  -7.41120149  -9.71378658  -9.12599992  -7.51656201
 -10.70703836  -9.71378658  -9.60842607  -4.52082973 -10.70703836
  -5.51408151  -9.68538711 -20.83514173  -4.52082973]
[9.99886732e-04 3.18627460e-03 2.42819281e-03 4.74148006e-06
 3.93367235e-03 8.99330669e-04 2.18537353e-02 2.18537353e-03
 3.93367235e-03 1.96683617e-02 8.09397602e-04 2.18537353